# MongoDB Landing Zone — Exploration

Connection: **MongoDB Atlas** (`mongo-mk1`, eu-central-1 Frankfurt)  
Database: `airline_landing`  
User: `airline-reader-ro` (read-only) via `MONGO_URI` from `.env`

| Collection | Populated by | Content |
|---|---|---|
| `adsb_raw` | `collectors/adsb_collector.py` | ADS-B snapshots (adsb.lol, Berlin 60 nm radius) |
| `opensky_raw` | `collectors/opensky_collector.py` | Departures & arrivals per airport (OpenSky Network) |
| `flight_tracker_raw` | `collectors/flight_tracker.py` | Single aircraft position points (tracked) |

**Sections 1–8:** ADS-B exploration  
**Sections 9–11:** OpenSky exploration + cross-collection join  
**Sections 12–14:** flight_tracker_raw exploration

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from datetime import timezone

sys.path.insert(0, '.')
from db.mongo.connector import from_env

db = from_env().connect()
print("Connected to MongoDB")

## 1. Collections and document counts

In [ ]:
client = db._client
database = client["airline_landing"]

print("Collections in airline_landing:\n")
for name in sorted(database.list_collection_names()):
    count = database[name].count_documents({})
    latest = database[name].find_one({}, {"collected_at": 1, "_id": 0}, sort=[("collected_at", -1)])
    latest_ts = latest.get("collected_at", "?")[:19] if latest else "?"
    print(f"  {name:<20}  docs={count:>5}  latest={latest_ts}")

col = db.collection("adsb_raw")

## 2. Latest snapshots (excluding `ac` array)

In [ ]:
recent = list(col.find({}, {'ac': 0}).sort('collected_at', -1).limit(10))

df_snaps = pd.DataFrame([
    {
        'collected_at': d['collected_at'],
        'total': d['total'],
        'source': d['source'],
        '_id': str(d['_id']),
    }
    for d in recent
])
print(df_snaps.to_string(index=False))

## 3. Letzten Snapshot als DataFrame aufklappen

In [ ]:
latest = col.find_one({}, sort=[('collected_at', -1)])
print(f"Snapshot: {latest['collected_at']}  —  {latest['total']} aircraft")

df = pd.DataFrame(latest['ac'])
print(f"Shape: {df.shape}")
print(f"Spalten: {list(df.columns)}")

In [ ]:
# readable overview
cols = [c for c in ['hex', 'flight', 'r', 't', 'alt_baro', 'gs', 'lat', 'lon', 'dst'] if c in df.columns]
print(df[cols].to_string(index=False))

## 4. Data quality — completeness of core fields

In [ ]:
key_fields = ['hex', 'flight', 'lat', 'lon', 'alt_baro', 'gs', 'r', 't']
present = {f: df[f].notna().sum() if f in df.columns else 0 for f in key_fields}
for field, count in present.items():
    pct = count / len(df) * 100
    print(f"  {field:<12} {count:>3}/{len(df)}  ({pct:.0f}%)")

## 5. alt_baro: number vs. 'ground'

In [ ]:
if 'alt_baro' in df.columns:
    on_ground = (df['alt_baro'] == 'ground').sum()
    airborne  = (df['alt_baro'] != 'ground').sum()
    print(f"On ground:  {on_ground}")
    print(f"Airborne:   {airborne}")

    df['alt_baro_ft'] = pd.to_numeric(df['alt_baro'], errors='coerce')
    df['on_ground']   = df['alt_baro'] == 'ground'

## 6. Top callsigns over Berlin

In [ ]:
if 'flight' in df.columns:
    top = (
        df['flight']
        .str.strip()
        .replace('', pd.NA)
        .dropna()
        .value_counts()
        .head(20)
    )
    print(top.to_string())

## 7. Aggregate all snapshots — aircraft count time series

In [ ]:
all_snaps = list(col.find({}, {'ac': 0}).sort('collected_at', 1))

df_ts = pd.DataFrame([
    {'collected_at': d['collected_at'], 'total': d['total']}
    for d in all_snaps
])
df_ts['collected_at'] = pd.to_datetime(df_ts['collected_at'], utc=True)

print(df_ts.to_string(index=False))
print(f"\nMean: {df_ts['total'].mean():.1f} aircraft/snapshot")
print(f"Min/Max: {df_ts['total'].min()} / {df_ts['total'].max()}")

---
## 9. OpenSky Raw — overview

Each document = one API call (departures **or** arrivals) for an airport and time window.

In [ ]:
from datetime import datetime, timezone

osky_col = db.collection("opensky_raw")

docs = list(osky_col.find({}, {"flights": 0}).sort("collected_at", -1))

df_osky = pd.DataFrame([
    {
        "collected_at": d["collected_at"][:19],
        "endpoint":     d["endpoint"],
        "airport":      d["query_params"]["airport"],
        "flight_count": d["flight_count"],
        "window_h":     round((d["query_params"]["end"] - d["query_params"]["begin"]) / 3600),
        "_id":          str(d["_id"]),
    }
    for d in docs
])

print(f"Dokumente in opensky_raw: {len(df_osky)}\n")
print(df_osky.to_string(index=False))

## 10. OpenSky flights — expand all documents

All flights from all opensky_raw documents as a flat DataFrame.

In [ ]:
rows = []
for doc in osky_col.find({}).sort("collected_at", 1):
    for f in doc.get("flights", []):
        rows.append({
            "icao24":    f.get("icao24", ""),
            "callsign":  (f.get("callsign") or "").strip(),
            "dep":       f.get("estDepartureAirport") or "?",
            "arr":       f.get("estArrivalAirport") or "?",
            "firstSeen": datetime.fromtimestamp(f["firstSeen"], timezone.utc).strftime("%Y-%m-%d %H:%M") if f.get("firstSeen") else None,
            "lastSeen":  datetime.fromtimestamp(f["lastSeen"],  timezone.utc).strftime("%H:%M")          if f.get("lastSeen")  else None,
            "endpoint":  doc["endpoint"],
            "airport":   doc["query_params"]["airport"],
        })

df_flights = pd.DataFrame(rows).drop_duplicates(subset=["icao24", "firstSeen"])
print(f"Unique flights (deduplicated): {len(df_flights)}\n")
print(df_flights.head(20).to_string(index=False))

## 11. Cross-collection join: ADS-B ↔ OpenSky

Join key: `adsb_raw.ac[].hex` = `opensky_raw.flights[].icao24` (ICAO24 transponder address)

- **ADS-B** provides: real-time position, altitude, speed, aircraft type  
- **OpenSky** adds: departure/destination airport, callsign, departure time  

Note: a match is only possible if an aircraft was visible in the ADS-B snapshot **and** captured in the OpenSky time window.

In [ ]:
# Build OpenSky index: icao24 → best available flight info
opensky_index = {}
for doc in osky_col.find({}):
    for f in doc.get("flights", []):
        icao = (f.get("icao24") or "").lower()
        if icao and icao not in opensky_index:
            opensky_index[icao] = {
                "callsign_osky": (f.get("callsign") or "").strip(),
                "dep":           f.get("estDepartureAirport") or "?",
                "arr":           f.get("estArrivalAirport") or "?",
                "firstSeen":     datetime.fromtimestamp(f["firstSeen"], timezone.utc).strftime("%H:%M UTC") if f.get("firstSeen") else "?",
            }

# Join with the latest ADS-B snapshot
latest_adsb = db.collection("adsb_raw").find_one({}, sort=[("collected_at", -1)])
snapshot_time = latest_adsb["collected_at"][:19]

matches = []
for ac in latest_adsb.get("ac", []):
    hex_id = (ac.get("hex") or "").lower()
    if hex_id in opensky_index:
        osky = opensky_index[hex_id]
        matches.append({
            "icao24":    hex_id,
            "callsign":  (ac.get("flight") or "").strip() or osky["callsign_osky"],
            "type":      ac.get("t", "?"),
            "alt_ft":    ac.get("alt_baro", "?"),
            "gs_kts":    round(ac.get("gs") or 0),
            "lat":       round(ac.get("lat", 0), 3) if ac.get("lat") else "?",
            "lon":       round(ac.get("lon", 0), 3) if ac.get("lon") else "?",
            "dep":       osky["dep"],
            "arr":       osky["arr"],
            "firstSeen": osky["firstSeen"],
        })

df_joined = pd.DataFrame(matches).sort_values("callsign")

print(f"ADS-B snapshot: {snapshot_time}  ({latest_adsb['total']} aircraft)")
print(f"OpenSky icao24s in index: {len(opensky_index)}")
print(f"Matches: {len(df_joined)}\n")
print(df_joined.to_string(index=False))

## 15. Close connection

---
## 12. flight_tracker_raw — overview

Each document = one tracked aircraft position (not a snapshot array).  
Fields: `flight_label`, `hex`, `registration`, `aircraft_type`, `lat`, `lon`, `alt_baro`, `gs`, `squawk`.

In [ ]:
ft_col = db.collection("flight_tracker_raw")

total_ft = ft_col.count_documents({})
latest_ft = ft_col.find_one({}, sort=[("collected_at", -1)])
oldest_ft = ft_col.find_one({}, sort=[("collected_at",  1)])
print(f"Total documents: {total_ft}")
print(f"Oldest entry:    {oldest_ft['collected_at'][:19]}")
print(f"Latest entry:    {latest_ft['collected_at'][:19]}")

docs_ft = list(ft_col.find({}))
df_ft = pd.DataFrame([
    {
        "collected_at":  d["collected_at"][:19],
        "flight_label":  d.get("flight_label", ""),
        "hex":           d.get("hex", ""),
        "registration":  d.get("registration", ""),
        "aircraft_type": d.get("aircraft_type", ""),
        "alt_baro":      d.get("alt_baro", ""),
        "gs":            d.get("gs", ""),
        "lat":           round(d["lat"], 3) if d.get("lat") else None,
        "lon":           round(d["lon"], 3) if d.get("lon") else None,
        "squawk":        d.get("squawk", ""),
    }
    for d in docs_ft
])
print(f"\nShape: {df_ft.shape}")
print(df_ft.head(10).to_string(index=False))

## 13. Aggregation — aircraft types and most active flights

Which types appear most often? Which `flight_label` values have the most entries?

In [ ]:
# Aircraft types
type_counts = df_ft["aircraft_type"].replace("", pd.NA).dropna().value_counts()
print("Aircraft types:")
print(type_counts.to_string())

print()

# Most active flights (flight_label)
flight_counts = df_ft["flight_label"].replace("", pd.NA).dropna().value_counts().head(15)
print("Most active flights (flight_label, top 15):")
print(flight_counts.to_string())

# MongoDB aggregation for comparison
pipeline = [
    {"$group": {"_id": "$aircraft_type", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}},
    {"$limit": 10},
]
print("\nMongoDB $group aircraft_type:")
for r in ft_col.aggregate(pipeline):
    print(f"  {r['_id'] or '(empty)':<10}  {r['count']:>4}")

## 14. Position trail of a single aircraft

All entries for a given `hex` (ICAO24) sorted chronologically.

In [ ]:
# Take the hex with the most entries
top_hex = df_ft["hex"].value_counts().idxmax()
track_docs = list(ft_col.find({"hex": top_hex}).sort("collected_at", 1))

df_track = pd.DataFrame([
    {
        "collected_at": d["collected_at"][:19],
        "flight_label": d.get("flight_label", ""),
        "alt_baro":     d.get("alt_baro", ""),
        "gs":           d.get("gs", ""),
        "lat":          round(d["lat"], 4) if d.get("lat") else None,
        "lon":          round(d["lon"], 4) if d.get("lon") else None,
    }
    for d in track_docs
])

print(f"hex: {top_hex}  —  {len(df_track)} entries")
print(df_track.to_string(index=False))

In [ ]:
db.close()
print("Connection closed.")